In [2]:
#Import libararies
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [3]:
# --- Setup ---
desktop = Path.home() / "Desktop"
output_dir = desktop / "data_presentation_outputs"
output_dir.mkdir(exist_ok=True)

In [4]:
# Load clean datasets
deals_df = pd.read_csv(desktop / "clean_deals.csv")
panel_df = pd.read_csv(desktop / "clean_firm_panel.csv")

print("=" * 70)
print("DATA QUALITY REPORT - CLEAN DEALS DATASET")
print("=" * 70)

DATA QUALITY REPORT - CLEAN DEALS DATASET


In [5]:
# 1. Basic info
print("\n1. Dataset shape:")
print(f"   Rows: {deals_df.shape[0]:,}, Columns: {deals_df.shape[1]}")

print("\n2. Column names and dtypes:")
print(deals_df.dtypes.to_string())



1. Dataset shape:
   Rows: 123, Columns: 10

2. Column names and dtypes:
deal_id              object
year                  int64
bvd_id               object
target_bvd_id        object
family_owned          int64
diversification       int64
deal_value_eur_m    float64
currency             object
avg_rate_to_eur     float64
cross_border          int64


In [6]:
# 2. Missing values
print("\n3. Missing values (should be 0 for clean data):")
missing = deals_df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "   No missing values found.")



3. Missing values (should be 0 for clean data):
target_bvd_id       3
deal_value_eur_m    5
currency            5
dtype: int64


In [7]:
# 3. Summary statistics for numeric variables
numeric_cols = deals_df.select_dtypes(include=[np.number]).columns.tolist()
print("\n4. Summary statistics (numeric variables):")
print(deals_df[numeric_cols].describe().round(2))


4. Summary statistics (numeric variables):
          year  family_owned  diversification  deal_value_eur_m  \
count   123.00        123.00           123.00            118.00   
mean   2019.98          0.46             0.85            126.10   
std       3.36          0.50             0.35            123.32   
min    2015.00          0.00             0.00              1.05   
25%    2017.00          0.00             1.00             31.30   
50%    2020.00          0.00             1.00            101.63   
75%    2023.00          1.00             1.00            175.32   
max    2025.00          1.00             1.00            678.19   

       avg_rate_to_eur  cross_border  
count           123.00        123.00  
mean              0.92          0.48  
std               0.31          0.50  
min               0.09          0.00  
25%               1.00          0.00  
50%               1.00          0.00  
75%               1.00          1.00  
max               1.28          1.00  


In [8]:
# 4. First 5 rows
print("\n5. First 5 rows (sample):")
print(deals_df.head(5).to_string())



5. First 5 rows (sample):
  deal_id  year       bvd_id target_bvd_id  family_owned  diversification  deal_value_eur_m currency  avg_rate_to_eur  cross_border
0  D00001  2025  BVD00000008   BVD00000054             1                1         75.158314      USD           1.1054             0
1  D00002  2018  BVD00000033   BVD00000128             0                1        197.510000      EUR           1.0000             0
2  D00003  2021  BVD00000022   BVD00000086             0                0        136.940000      EUR           1.0000             1
3  D00004  2023  BVD00000008   BVD00000099             1                1         25.440000      EUR           1.0000             0
4  D00005  2019  BVD00000006   BVD00000080             0                0         19.639842      CHF           1.0662             0


In [9]:
# 5. Frequency tables for categorical variables
categorical_cols = ['family_owned', 'diversification', 'cross_border']
print("\n6. Frequency tables:")
for col in categorical_cols:
    if col in deals_df.columns:
        print(f"\n   {col}:")
        print(deals_df[col].value_counts().to_string())


6. Frequency tables:

   family_owned:
family_owned
0    66
1    57

   diversification:
diversification
1    105
0     18

   cross_border:
cross_border
0    64
1    59


In [12]:
# 7. Deal value distribution histogram
if 'deal_value_eur_m' in deals_df.columns:
    plt.figure(figsize=(10, 6))
    plt.hist(deals_df['deal_value_eur_m'], bins=50, edgecolor='black', alpha=0.7)
    plt.xlabel('Deal Value (EUR millions)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Deal Values (Top 99th percentile shown)')
    cap = deals_df['deal_value_eur_m'].quantile(0.99)
    plt.xlim(0, cap)
    plt.grid(True, alpha=0.3)
    plt.savefig(output_dir / 'deal_value_histogram.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\n7. Histogram saved to: {output_dir / 'deal_value_histogram.png'}")
else:
    print("\n7. Skipping histogram: 'deal_value_eur_m' column not found.")



7. Histogram saved to: C:\Users\user\Desktop\data_presentation_outputs\deal_value_histogram.png


In [13]:
# 8. Deal count and total value by year
if 'year' in deals_df.columns and 'deal_value_eur_m' in deals_df.columns:
    yearly = deals_df.groupby('year').agg(
        n_deals=('deal_id', 'count'),
        total_value=('deal_value_eur_m', 'sum')
    ).reset_index()
    
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax2 = ax1.twinx()
    ax1.bar(yearly['year'], yearly['n_deals'], alpha=0.7, color='steelblue', label='Number of deals')
    ax2.plot(yearly['year'], yearly['total_value'], color='darkred', marker='o', linewidth=2, label='Total value (EUR mil)')
    ax1.set_xlabel('Year')
    ax1.set_ylabel('Number of deals', color='steelblue')
    ax2.set_ylabel('Total deal value (EUR millions)', color='darkred')
    plt.title('Deal Activity by Year')
    plt.grid(True, alpha=0.2)
    fig.tight_layout()
    plt.savefig(output_dir / 'deals_by_year.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"8. Yearly trends saved to: {output_dir / 'deals_by_year.png'}")
else:
    print("\n8. Skipping yearly trends: required columns missing.")

8. Yearly trends saved to: C:\Users\user\Desktop\data_presentation_outputs\deals_by_year.png


In [14]:
# 9. Correlation heatmap (for numeric controls)
if len(numeric_cols) >= 2:
    plt.figure(figsize=(10, 8))
    corr = deals_df[numeric_cols].corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
    plt.title('Correlation Matrix - Key Numeric Variables')
    plt.tight_layout()
    plt.savefig(output_dir / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"9. Correlation heatmap saved to: {output_dir / 'correlation_heatmap.png'}")
else:
    print("\n9. Skipping heatmap: less than 2 numeric columns.")


9. Correlation heatmap saved to: C:\Users\user\Desktop\data_presentation_outputs\correlation_heatmap.png


In [15]:
# 10. Firm-year panel summary
print("\n" + "=" * 70)
print("FIRM-YEAR PANEL SUMMARY")
print("=" * 70)
print(f"\nPanel dimensions: {panel_df.shape[0]:,} rows, {panel_df.shape[1]} columns")
if 'bvd_id' in panel_df.columns:
    print(f"Unique firms: {panel_df['bvd_id'].nunique():,}")
if 'year' in panel_df.columns:
    print(f"Year range: {panel_df['year'].min()} to {panel_df['year'].max()}")



FIRM-YEAR PANEL SUMMARY

Panel dimensions: 550 rows, 8 columns
Unique firms: 50
Year range: 2015 to 2025


In [16]:
# Summary of key panel variables
panel_numeric = ['n_acq', 'total_acq_value_eur_m', 'total_assets_eur_m', 'leverage', 'roa']
existing = [c for c in panel_numeric if c in panel_df.columns]
if existing:
    print("\nPanel summary statistics:")
    print(panel_df[existing].describe().round(2))
else:
    print("\nNo expected numeric panel variables found. Available columns:", panel_df.columns.tolist())

print("\n" + "=" * 70)
print(f"✅ All outputs saved to: {output_dir}")
print("   Include these images and console outputs in your proposal.")
print("=" * 70)


Panel summary statistics:
        n_acq  total_acq_value_eur_m  total_assets_eur_m  leverage     roa
count  550.00                 550.00              550.00    550.00  550.00
mean     0.22                  27.05              493.61      0.41    0.05
std      0.47                  79.80              526.52      0.17    0.10
min      0.00                   0.00                2.54      0.10   -0.22
25%      0.00                   0.00              121.64      0.26   -0.01
50%      0.00                   0.00              327.18      0.41    0.06
75%      0.00                   0.00              676.20      0.55    0.12
max      3.00                 678.19             3720.86      0.70    0.31

✅ All outputs saved to: C:\Users\user\Desktop\data_presentation_outputs
   Include these images and console outputs in your proposal.
